# Notebook 04a — Rebuild Day-2 Agent with LangChain

## Objective

Rebuild the Day-2 agent using LangChain while keeping the same:

- LLM provider
- LLM model
- Calculator tool
- Document lookup tool
- Mock database query tool
- Test scenarios

The purpose is to compare a raw Python agent with a
framework-based LangChain agent.

## Learning Goals

- Understand LangChain model abstraction
- Define tools using LangChain
- Use structured tool arguments
- Bind tools to an LLM
- Build an agent
- Execute tool calls
- Compare LangChain with the raw Day-2 implementation

In [5]:
%pip install -U langchain langchain-openai

  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.17
    Uninstalling langchain-1.3.17:
      Successfully uninstalled langchain-1.3.17
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import langchain
import langchain_openai

print("LangChain version:", langchain.__version__)
print("langchain-openai version:", langchain_openai.__version__)

LangChain version: 1.4.2
langchain-openai version: 1.6.5


## 1. Imports

LangChain provides abstractions for:

- LLMs
- tools
- agents
- tool calling

We will also use Pydantic for structured tool arguments.

In [11]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain.tools import tool

print("Imports successful.")

Imports successful.


In [12]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set.")

MODEL = "openai/gpt-oss-20b"

model = ChatOpenAI(
    model=MODEL,
    temperature=0,
    api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

print("LangChain model configured successfully.")

LangChain model configured successfully.


In [13]:
response = model.invoke("What is RAG?")

print(response.content)

## RAG – Retrieval‑Augmented Generation

**RAG** stands for **Retrieval‑Augmented Generation**.  
It’s a modern approach to building language‑model‑based systems that combine the strengths of two complementary components:

| Component | What it does | Typical technology |
|-----------|--------------|--------------------|
| **Retriever** | Finds relevant documents, passages, or facts from a large knowledge base (or the web) that are likely to answer a user’s query. | Dense vector search (e.g., FAISS), BM25, or hybrid retrieval. |
| **Generator** | A language model (e.g., GPT‑4, T5, BART) that takes the retrieved evidence and the original query, then produces a fluent, context‑aware answer. | Transformer‑based seq2seq models, fine‑tuned on QA data. |

### Why RAG?

| Problem | Traditional LLMs | RAG |
|---------|------------------|-----|
| **Knowledge cutoff** | Static weights; can’t “learn” new facts after training. | The retriever can pull in up‑to‑date documents at inference time. |
|

## 2. Calculator Tool

The calculator is the first tool from the Day-2 agent.

LangChain will expose the Python function as a tool that the
LLM can request.

In [14]:
class CalculatorArgs(BaseModel):
    expression: str = Field(
        ...,
        description="Mathematical expression to calculate"
    )


def calculator(expression: str) -> float:
    """
    Calculate a simple mathematical expression.
    """
    allowed_chars = "0123456789+-*/(). "

    if not all(char in allowed_chars for char in expression):
        raise ValueError(
            "Expression contains unsupported characters."
        )

    return eval(
        expression,
        {"__builtins__": {}},
        {}
    )

In [16]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""

    allowed_chars = "0123456789+-*/(). "

    if not all(char in allowed_chars for char in expression):
        return "Error: expression contains unsupported characters."

    try:
        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [17]:
print("Tool name:", calculator.name)
print("Description:", calculator.description)
print("Schema:", calculator.args_schema)

Tool name: calculator
Description: Calculate a mathematical expression.
Schema: <class 'langchain_core.utils.pydantic.calculator'>


In [18]:
result = calculator.invoke(
    {"expression": "25 * 47"}
)

print("Result:", result)

Result: 1175


## Build the LangChain Agent

In the Day-2 raw agent, we manually implemented the agent loop:

1. Receive the user request.
2. Decide whether a tool is required.
3. Select a tool.
4. Execute the tool.
5. Observe the result.
6. Continue until a final answer is produced.

LangChain provides a higher-level agent abstraction that manages this tool-calling loop for us.

We will use the same three tools:

- calculator
- document_lookup
- mock_db_query

The model and provider will also remain the same as the raw implementation.

In [19]:
import langchain.agents as agents

print("Available agent APIs:")
print([name for name in dir(agents) if not name.startswith("_")])

Available agent APIs:
['AgentState', 'create_agent', 'factory', 'middleware', 'structured_output']


print(type(tools))
print(tools)

## Document Lookup Tool

This tool provides a small in-memory knowledge base for testing
the agent's document-retrieval behavior.

It is intentionally simple because the goal of 04a is to compare
the agent framework, not to build a production RAG system.

In [24]:
DOCUMENTS = {
    "docuchat": """
    DocuChat is a RAG-powered chat application.
    It uses PostgreSQL with pgvector to store document embeddings.
    Relevant document chunks are retrieved using semantic similarity.
    """,

    "rag": """
    Retrieval-Augmented Generation combines document retrieval
    with language model generation. Relevant context is retrieved
    before the answer is generated.
    """,

    "agents": """
    An AI agent can decide which action to take, use tools,
    observe results, and continue until the task is complete.
    """
}

In [25]:
@tool
def document_lookup(query: str) -> str:
    """Search the document knowledge base using a keyword query."""

    query_lower = query.lower()
    matches = []

    for name, content in DOCUMENTS.items():
        if name in query_lower:
            matches.append(content.strip())

    if not matches:
        return "No relevant document found."

    return "\n\n".join(matches)

In [26]:
result = document_lookup.invoke(
    {"query": "Tell me about DocuChat"}
)

print(result)

DocuChat is a RAG-powered chat application.
    It uses PostgreSQL with pgvector to store document embeddings.
    Relevant document chunks are retrieved using semantic similarity.


## Mock Database Tool

This tool simulates a database query.

The Day-2 agent used a mock database so that we could focus on
agent behavior without requiring a real database connection.

In [27]:
MOCK_DB = {
    "users": [
        {"id": 1, "name": "Arun", "email": "arun@example.com"},
        {"id": 2, "name": "Raj", "email": "raj@example.com"}
    ],

    "documents": [
        {"id": 1, "filename": "docuchat.txt", "chunks": 5},
        {"id": 2, "filename": "rag_notes.txt", "chunks": 8}
    ]
}

In [28]:
@tool
def mock_db_query(table: str, limit: int = 5) -> str:
    """Query the mock database and return rows from a table."""

    if table not in MOCK_DB:
        return f"Error: unknown table '{table}'."

    rows = MOCK_DB[table][:limit]

    return str(rows)

In [29]:
result = mock_db_query.invoke(
    {
        "table": "users",
        "limit": 2
    }
)

print(result)

[{'id': 1, 'name': 'Arun', 'email': 'arun@example.com'}, {'id': 2, 'name': 'Raj', 'email': 'raj@example.com'}]


In [30]:
agent_tools = [
    calculator,
    document_lookup,
    mock_db_query
]

print("Agent tools:")
for item in agent_tools:
    print("-", item.name)

Agent tools:
- calculator
- document_lookup
- mock_db_query


In [31]:
agent = agents.create_agent(
    model=model,
    tools=agent_tools
)

print("LangChain agent created successfully.")

LangChain agent created successfully.


## Test 1 — Calculator

The LangChain agent should recognize that this request requires
the calculator tool.

Expected flow:

User request
    ↓
LangChain Agent
    ↓
calculator
    ↓
Tool result
    ↓
Final answer

In [34]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate 25 * 47."
            }
        ]
    }
)

print("Final answer:")
print(result["messages"][-1].content)

Final answer:
The result of \(25 \times 47\) is **1175**.


## Inspect Agent Execution

The raw Day-2 agent explicitly implemented the agent loop.

LangChain manages this loop internally.

We can inspect the returned messages to see the sequence of
user message, tool call, tool result, and final response.

In [35]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate 125 * 48."
            }
        ]
    }
)

for i, message in enumerate(result["messages"], start=1):
    print("=" * 70)
    print(f"MESSAGE {i}")
    print("TYPE:", type(message).__name__)
    print("CONTENT:", message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("TOOL CALLS:")
        print(message.tool_calls)

MESSAGE 1
TYPE: HumanMessage
CONTENT: Calculate 125 * 48.
MESSAGE 2
TYPE: AIMessage
CONTENT: 125 × 48 = **6 000**


## Test 2 — Document Lookup

The agent should identify that the question requires information
from the document knowledge base.

It should therefore use the `document_lookup` tool.

In [36]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is DocuChat?"
            }
        ]
    }
)

print("Final answer:")
print(result["messages"][-1].content)

Final answer:
**DocuChat** is a RAG‑powered (Retrieval‑Augmented Generation) chat application.  
It lets users ask questions about a collection of documents and returns answers that are grounded in the actual content of those documents.

Key technical points:

| Feature | How it works |
|---------|--------------|
| **Document storage** | Uses PostgreSQL with the `pgvector` extension to store dense vector embeddings of document chunks. |
| **Retrieval** | When a user submits a query, the system embeds the query, then performs a semantic similarity search against the stored vectors to pull the most relevant document fragments. |
| **Generation** | The retrieved chunks are fed into a language model (e.g., GPT‑4) as context, so the model can generate an answer that is both fluent and factually tied to the source material. |
| **RAG workflow** | Retrieval → Contextualization → Generation → Response. |

In short, DocuChat combines vector‑based semantic search with large‑language‑model genera

## Test 3 — Mock Database

The agent should select the `mock_db_query` tool when the user
asks for database information.

In [37]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Show me the users in the database."
            }
        ]
    }
)

print("Final answer:")
print(result["messages"][-1].content)

Final answer:
Here are the users currently stored in the database:

| ID | Name | Email |
|----|------|-------------------|
| 1  | Arun | arun@example.com |
| 2  | Raj  | raj@example.com |

Let me know if you’d like more details or any other queries!


In [38]:
test_questions = [
    "What is 25 * 17?",
    "What is DocuChat?",
    "Show me the users in the database."
]

for question in test_questions:
    print("=" * 70)
    print("QUESTION:", question)

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    print("ANSWER:", result["messages"][-1].content)

QUESTION: What is 25 * 17?
ANSWER: 25 × 17 = **425**
QUESTION: What is DocuChat?
ANSWER: **DocuChat** is a RAG‑powered (Retrieval‑Augmented Generation) chat application.  
It lets users ask questions about a collection of documents and returns answers that are grounded in the actual content of those documents.

Key technical points:

| Feature | How it works |
|---------|--------------|
| **Document storage** | Uses PostgreSQL with the `pgvector` extension to store dense vector embeddings of document chunks. |
| **Retrieval** | When a user submits a query, the system embeds the query, then performs a semantic similarity search against the stored vectors to pull the most relevant document fragments. |
| **Generation** | The retrieved chunks are fed into a language model (e.g., GPT‑4) as context, so the model can generate an answer that is both fluent and factually tied to the source material. |
| **RAG workflow** | Retrieval → Contextualization → Generation → Response. |

In short, Docu

## Test 4 — Ambiguous Request

Agents must handle incomplete requests instead of assuming
missing information.

Example:

"Calculate this."

The user has not provided an expression.

Observe how the LangChain agent handles this request.

In [39]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate this."
            }
        ]
    }
)

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
I’m happy to help! Could you please provide the expression or problem you’d like me to calculate?


## Test 5 — Unsupported Request

Test a request that does not match the capabilities of the
available tools.

The goal is to observe whether the agent avoids pretending that
it can perform an unavailable operation.

In [40]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Book me a flight to London tomorrow."
            }
        ]
    }
)

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
I’m sorry, but I don’t have the ability to book flights directly. However, I can help you find flight options or suggest travel agencies that can handle the booking for you. Let me know how you’d like to proceed!


## Raw Agent vs LangChain Agent

Both implementations use the same:

- LLM provider
- LLM model
- calculator tool
- document lookup tool
- mock database tool

The main difference is the implementation of the agent runtime.

The Day-2 raw agent exposes the internal agent loop directly.

The LangChain implementation provides a higher-level abstraction
that manages the tool-calling workflow.

In [41]:
comparison = [
    ("LLM Provider", "Groq", "Groq"),
    ("Model", "openai/gpt-oss-20b", "openai/gpt-oss-20b"),
    ("Calculator", "Python function", "LangChain @tool"),
    ("Document Lookup", "Python function", "LangChain @tool"),
    ("Mock DB", "Python function", "LangChain @tool"),
    ("Tool Schema", "Manual/Pydantic", "Inferred by LangChain"),
    ("Agent Loop", "Manually implemented", "Framework managed"),
    ("Tool Execution", "Manually implemented", "Framework managed"),
    ("Abstraction", "Low-level", "Higher-level"),
    ("Framework Dependency", "Minimal", "LangChain"),
]

print(f"{'Feature':<25} {'Day-2 Raw Agent':<30} {'04a LangChain':<30}")
print("-" * 90)

for feature, raw, langchain_version in comparison:
    print(f"{feature:<25} {raw:<30} {langchain_version:<30}")

Feature                   Day-2 Raw Agent                04a LangChain                 
------------------------------------------------------------------------------------------
LLM Provider              Groq                           Groq                          
Model                     openai/gpt-oss-20b             openai/gpt-oss-20b            
Calculator                Python function                LangChain @tool               
Document Lookup           Python function                LangChain @tool               
Mock DB                   Python function                LangChain @tool               
Tool Schema               Manual/Pydantic                Inferred by LangChain         
Agent Loop                Manually implemented           Framework managed             
Tool Execution            Manually implemented           Framework managed             
Abstraction               Low-level                      Higher-level                  
Framework Dependency      Min

## Reflection

### What did we learn?

The Day-2 implementation showed how an agent works internally by
manually implementing the agent loop and tool execution.

LangChain provides abstractions that reduce the amount of code
required to build a tool-using agent.

### What became easier?

- Tool registration
- Tool schema generation
- Connecting tools to the model
- Agent creation
- Tool execution
- Agent loop management

### What is the trade-off?

LangChain hides some implementation details behind abstractions.

This can make common agent development easier, but understanding
the underlying framework is still important for debugging and
customization.

### Key takeaway

The raw implementation helps us understand agent fundamentals.

LangChain helps us implement those patterns using reusable
framework abstractions.